In [ ]:
import os

In [ ]:
import torch
import ale_py
import gymnasium as gym

In [ ]:
# On windows you might need to change the working dir with the following command to be able to import the project
%cd ..

In [ ]:
from project.environments import BaseWrapper
from project.agents import DQNAgent
from project.environments.loops import TrainingLoop

In [ ]:
gym.register_envs(ale_py)

In [ ]:
PATH = os.path.join("parameters", "test_loading.pth")
FRAMES = 1_000
LR = 25e-4
DISCOUNT = 0.99
REPLAY_SIZE = 500_000
N_ACTIONS = 4
ENVIRONMENT = "ALE/Breakout-v5"
TARGET_UPDATE_FREQUENCY = 10_000
BATCH_SIZE = 32
FINAL_EXPLORATION_FRAME = 1_000_000
REPLAY_START_SIZE = 50_000

In [ ]:
device = "cpu"
if torch.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
print(f"Using: {device}")

In [ ]:
env = BaseWrapper.create_environment("ALE/Breakout-v5")

In [ ]:
agent = DQNAgent(
    train=True,
    lr=LR,
    discount=DISCOUNT,
    replay_size=REPLAY_SIZE,
    n_actions=N_ACTIONS,
    target_update_frequency=TARGET_UPDATE_FREQUENCY,
    batch_size=BATCH_SIZE,
    final_exploration_frame=FINAL_EXPLORATION_FRAME,
    replay_start_size=REPLAY_START_SIZE,
    obs_shape=env.observation_space.shape
).to(device)
if os.path.exists(PATH):
    agent.load(PATH)
    print("Loaded existing checkpoint!")

In [ ]:
loop = TrainingLoop(
    env=env,
    agent=agent
)

In [ ]:
loop.run(FRAMES)
agent.save(PATH)